# Session 5 · Linear Regression: The Idea

**Machine Learning Foundations · Sanketana School of Code**

Last session you trained a model and it just *worked* — `fit`, `predict`, done. We called it magic. Today we open the box, and it turns out the magic is something you already know: **a straight line.**

By the end of this notebook you will be able to:

- say what makes a prediction problem **regression** (predicting a *number*)
- describe a line as two numbers — a **slope** and an **intercept**
- draw your own line of best fit by eye, then let sklearn draw its own
- read the slope and intercept as a **claim about the world**, and predict a new value

## Warm-up · Last session's homework

Your coach will walk through Session 4's housing capstone (about 10 minutes). Check: did you get a **test score** and write an **insight**?

That homework ended with a question: the model *drew a line* through all that housing data — but **how**, and what was that line actually saying? That question is today.

## From a black box to a line

Keep the workflow map in mind. Last session you walked all four steps; today we zoom into one — **model** — and finally understand what it built.

| Step | Session 4 | Today |
|---|---|---|
| data | load, clean, split | load anchor data, pick **one** feature |
| **model** | `fit` a sealed box | **open the box: it's a line** |
| evaluation | `.score()`, a mystery number | *(Session 8 — not yet)* |
| insight | nudge a feature | read the line's slope as a claim |

First, the word for today's job. When the thing we predict is a **number on a scale** — a test score, a price, a temperature — that's **regression**. When it's a *category* (spam / not spam), that's *classification* (Module 3). All of Module 2 is regression: **predicting a continuous value.**

## Step 1 · Meet the data — one feature at a time

We return to the anchor students. Last session we used all five habits at once. Today we use just **one** — `study_hours_per_week` — so we can *see* the whole model on a single 2-D picture.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")

# One feature (x) and the label (y).
feature = "study_hours_per_week"
label = "test_score"

print("students:", students.shape[0])
students[[feature, label]].head()

In [ ]:
# Scatter every student: study hours across, test score up.
plt.figure(figsize=(7, 5))
plt.scatter(students[feature], students[label], alpha=0.5)
plt.xlabel("study hours per week")
plt.ylabel("test score")
plt.title("Each dot is one student")
plt.show()

Look at the cloud: as study hours go up, scores tend to go up. It's not a perfect line — real students scatter — but there's a clear upward drift. Our whole job today is to draw the **one straight line** that best summarises that drift.

A straight line is just **two numbers**:

```
prediction  =  slope × feature  +  intercept
test_score  =  slope × study_hours  +  intercept
```

- **slope** — extra points per extra hour of study (how tilted the line is)
- **intercept** — the score the line predicts at zero study hours (where it starts)

## Step 2 · Draw the line by eye

Before we let the computer do it, *you* do it. Pick a **slope** and an **intercept** below, run the cell, and see your line drawn over the dots. Then change your two numbers and run again until the line looks like a good fit.

✏️ **This cell is yours** — the guess below is just a starting point. Change it and re-run.

In [ ]:
# ✏️ TODO: pick your own slope and intercept, then re-run to see the line move.
my_slope = 2.0        # extra points per study hour  (try 1, 3, 4 ...)
my_intercept = 40.0   # score at zero study hours    (try 20, 30, 45 ...)

# Draw the dots, then your line on top.
import numpy as np
x_line = np.array([students[feature].min(), students[feature].max()])
y_line = my_slope * x_line + my_intercept

plt.figure(figsize=(7, 5))
plt.scatter(students[feature], students[label], alpha=0.5, label="students")
plt.plot(x_line, y_line, color="red", linewidth=2, label="my line")
plt.xlabel("study hours per week")
plt.ylabel("test score")
plt.title(f"My line: score = {my_slope} × study_hours + {my_intercept}")
plt.legend()
plt.show()

### How wrong is your line?

Every dot that isn't *on* your line is a **miss** — the gap between the real score and what your line predicted. Let's measure a few of those gaps.

In [ ]:
# Your line's prediction for the first 5 students, next to the truth.
sample = students.head(5).copy()
sample["my_prediction"] = my_slope * sample[feature] + my_intercept
sample["miss"] = sample[label] - sample["my_prediction"]
sample[[feature, label, "my_prediction", "miss"]].round(1)

### ✏️ Reflect

1. Look at the `miss` column. Are your line's misses small, or is it consistently too high / too low? Nudge `my_slope` and `my_intercept` and try to make the misses smaller.
2. Could a friend pick a *different* slope and intercept that look about as good? How would the two of you decide whose line is actually better — without just eyeballing it?

*Your answers:*

1. 
2. 

## Step 3 · Let sklearn draw the line

You got close by eye. Now the machine. The same `LinearRegression` from last session — but with **one** feature, its line has just one slope and one intercept, so we can read them.

> **Note the double brackets:** `students[[feature]]` gives a one-column *table* (what sklearn wants for `X`), not a single column of values. This trips people up all the time.

In [ ]:
from sklearn.linear_model import LinearRegression

X = students[[feature]]   # double brackets → a one-column table
y = students[label]

# ✏️ TODO: create the model and fit it on the single feature.
model = LinearRegression()
model.fit(X, y)

sk_slope = model.coef_[0]      # coef_ is an array; one feature → take [0]
sk_intercept = model.intercept_
print(f"sklearn slope:     {sk_slope:.2f}  (points per study hour)")
print(f"sklearn intercept: {sk_intercept:.2f}  (score at zero study hours)")

In [ ]:
# Overlay: your line (red) vs sklearn's line (green).
y_mine = my_slope * x_line + my_intercept
y_sklearn = sk_slope * x_line + sk_intercept

plt.figure(figsize=(7, 5))
plt.scatter(students[feature], students[label], alpha=0.4, label="students")
plt.plot(x_line, y_mine, color="red", linewidth=2, label="my line")
plt.plot(x_line, y_sklearn, color="green", linewidth=2, label="sklearn line")
plt.xlabel("study hours per week")
plt.ylabel("test score")
plt.title("Your line vs the machine’s")
plt.legend()
plt.show()

Your line and sklearn's are probably **close but not identical**. So here's the question that should be bugging you:

> **Why is sklearn's line the “best” one, and not yours?**

We can't settle it by eye — that's exactly the point. To decide fairly we need a single **number** that says how wrong any line is. Building that number is **Session 6**. Today, just notice the question is real and we can't answer it by looking.

## Step 4 · Read the line as a claim about the world

This is the part to remember. A fitted line isn't a plot decoration — it's a **statement**. Read sklearn's two numbers out loud:

- **slope** ≈ the points added per extra hour of study
- **intercept** ≈ the score the line predicts for zero study hours

### ✏️ Interpret

Fill in the blanks using the numbers printed above.

1. “In this data, each extra hour of study per week is associated with about ____ more points on the test.”
2. The intercept is about ____ . In one sentence, what does it claim? (And why is it a bit of a stretch to treat it as a real fact about students?)

*Your answers:*

1. 
2. 

### Predicting is just arithmetic

To predict, the model doesn't look up a similar student — it plugs a number into `slope × x + intercept`. Let's prove it: predict for a student who studies **10 hours**, two ways, and check they match.

In [ ]:
study = 10

# Way 1: ask the model. Give it a one-row table, same shape as X.
scenario = pd.DataFrame({feature: [study]})
by_model = model.predict(scenario)[0]

# Way 2: do the line's arithmetic by hand.
by_hand = sk_slope * study + sk_intercept

print(f"model.predict:  {by_model:.1f}")
print(f"slope*x+intcpt: {by_hand:.1f}")
print("same number → .predict() is just evaluating the line.")

### ✏️ Stretch — a feature that pushes the other way

Not every habit lifts scores. Swap the feature to `screen_time_hours_per_day`, refit, and read the new slope. Run the cell, then answer below.

In [ ]:
stretch_feature = "screen_time_hours_per_day"

m2 = LinearRegression().fit(students[[stretch_feature]], students[label])
print(f"slope for {stretch_feature}: {m2.coef_[0]:.2f}")

plt.figure(figsize=(7, 5))
plt.scatter(students[stretch_feature], students[label], alpha=0.4)
xs = np.array([students[stretch_feature].min(), students[stretch_feature].max()])
plt.plot(xs, m2.coef_[0] * xs + m2.intercept_, color="green", linewidth=2)
plt.xlabel("screen time (hours per day)")
plt.ylabel("test score")
plt.title("A negative slope")
plt.show()

✏️ The slope for screen time is **negative**. Write one sentence saying what a negative slope *claims* about screen time and scores in this data.

*Your answer:* 

## What we learned

✏️ Three quick reflections — one line each:

1. Regression predicts a ________ (what kind of thing?).
2. A line is described by two numbers: ________ and ________.
3. The slope of the study-hours line means, in plain English: ________.

---

**You opened the box.** A `LinearRegression` model is just a line — `slope × feature + intercept` — and that line is a claim about the world you can read and predict from.

**But one thing is unfinished:** your line and sklearn's disagreed, and we couldn't say whose was better without guessing. **Session 6** fixes that — we'll build a single number for how wrong a line is (the *cost*), and watch the best line fall out as the one that makes it smallest.